This notebook is made to show and implement how data is downloaded via api call and also performed cleaning procedures.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys

import pandas as pd
import requests

ROOT = Path.cwd()

# Notebook notebooks/ folder ke andar ho toh parent project root hai.
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if not (ROOT / "requirements.txt").exists():
    raise FileNotFoundError("Notebook ko project root ya notebooks/ se kholo.")

print("Python:", sys.executable)
print("Project root:", ROOT)

Python: /Users/zsres/Documents/M.Tech/Statistical-Inference-Assignment/.venv/bin/python
Project root: /Users/zsres/Documents/M.Tech/Statistical-Inference-Assignment


In [2]:
COUNTRY_CODE = "NPL"
INDICATOR_CODE = "NY.GDP.PCAP.KD"

url = (
    f"https://api.worldbank.org/v2/country/"
    f"{COUNTRY_CODE}/indicator/{INDICATOR_CODE}"
)

params = {
    "source": 2,
    "date": "2010:2024",
    "format": "json",
    "per_page": 100,
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

payload = response.json()

# API kabhi HTTP 200 ke saath bhi error message bhej sakti hai.
if not (
    isinstance(payload, list)
    and len(payload) == 2
    and isinstance(payload[0], dict)
    and isinstance(payload[1], list)
):
    raise ValueError(f"Unexpected World Bank response: {payload}")

download_time = datetime.now(timezone.utc)
stamp = download_time.strftime("%Y%m%dT%H%M%S%fZ")

# Original response ka separate, timestamped copy.
raw_file = ROOT / "data" / "raw" / f"{INDICATOR_CODE}_{stamp}.json"
raw_file.write_bytes(response.content)

download_record = {
    "downloaded_at_utc": download_time.isoformat(),
    "url": response.url,
    "source_last_updated": payload[0].get("lastupdated"),
    "indicator_code": INDICATOR_CODE,
}

metadata_file = ROOT / "data" / "metadata" / f"download_{stamp}.json"
metadata_file.write_text(
    json.dumps(download_record, indent=2),
    encoding="utf-8",
)

print("Raw data saved:", raw_file.name)
print("Annual records returned:", len(payload[1]))

Raw data saved: NY.GDP.PCAP.KD_20261007T163511493221Z.json
Annual records returned: 15


In [3]:
COUNTRY_CODE = "NPL"
INDICATOR_CODE = "NY.GDP.PCAP.KD"

url = (
    f"https://api.worldbank.org/v2/country/"
    f"{COUNTRY_CODE}/indicator/{INDICATOR_CODE}"
)

params = {
    "source": 2,
    "date": "2010:2024",
    "format": "json",
    "per_page": 100,
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

payload = response.json()

# API kabhi HTTP 200 ke saath bhi error message bhej sakti hai.
if not (
    isinstance(payload, list)
    and len(payload) == 2
    and isinstance(payload[0], dict)
    and isinstance(payload[1], list)
):
    raise ValueError(f"Unexpected World Bank response: {payload}")

download_time = datetime.now(timezone.utc)
stamp = download_time.strftime("%Y%m%dT%H%M%S%fZ")

# Original response ka separate, timestamped copy.
raw_file = ROOT / "data" / "raw" / f"{INDICATOR_CODE}_{stamp}.json"
raw_file.write_bytes(response.content)

download_record = {
    "downloaded_at_utc": download_time.isoformat(),
    "url": response.url,
    "source_last_updated": payload[0].get("lastupdated"),
    "indicator_code": INDICATOR_CODE,
}

metadata_file = ROOT / "data" / "metadata" / f"download_{stamp}.json"
metadata_file.write_text(
    json.dumps(download_record, indent=2),
    encoding="utf-8",
)

print("Raw data saved:", raw_file.name)
print("Annual records returned:", len(payload[1]))

Raw data saved: NY.GDP.PCAP.KD_20261007T163650044805Z.json
Annual records returned: 15


This below cell is to convert observations data in the readable format

In [4]:
rows = [
    {
        "Year": int(observation["date"]),
        "GDP_per_capita": observation["value"],
    }
    for observation in payload[1]
]

gdp = (
    pd.DataFrame(rows)
    .set_index("Year")
    .reindex(range(2010, 2025))
    .reset_index()
)

gdp["Country"] = "Nepal"
gdp["Country_Code"] = COUNTRY_CODE

gdp = gdp[
    ["Year", "Country", "Country_Code", "GDP_per_capita"]
]

gdp.to_csv(
    ROOT / "data" / "processed" / "nepal_gdp_per_capita_2010_2024.csv",
    index=False,
)

print("Potential observations:", len(gdp))
print("Available observations:", gdp["GDP_per_capita"].notna().sum())
print("Missing observations:", gdp["GDP_per_capita"].isna().sum())

gdp

Potential observations: 15
Available observations: 15
Missing observations: 0


,Year,Country,Country_Code,GDP_per_capita
0,2010,Nepal,NPL,721.265221
1,2011,Nepal,NPL,742.617253
2,2012,Nepal,NPL,775.310578
3,2013,Nepal,NPL,801.039061
4,2014,Nepal,NPL,846.665542
5,2015,Nepal,NPL,875.543635
6,2016,Nepal,NPL,875.188944
7,2017,Nepal,NPL,951.857299
8,2018,Nepal,NPL,1021.914922
9,2019,Nepal,NPL,1077.117965


In [5]:
INDICATORS = {
    "GDP_per_capita": "NY.GDP.PCAP.KD",
    "GDP_growth": "NY.GDP.MKTP.KD.ZG",
    "Life_expectancy": "SP.DYN.LE00.IN",
    "Basic_drinking_water": "SH.H2O.BASW.ZS",
    "Secondary_school_enrollment": "SE.SEC.ENRR",
    "Electricity_access": "EG.ELC.ACCS.ZS",
    "Internet_users": "IT.NET.USER.ZS",
    "CO2_per_capita": "EN.ATM.CO2E.PC",
    "Forest_area": "AG.LND.FRST.ZS",
}

PARAMS = {
    "source": 2,
    "date": "2010:2024",
    "format": "json",
    "per_page": 100,
}

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

# Is batch ke original responses ek separate folder mein save honge.
RAW_DIR = ROOT / "data" / "raw" / RUN_ID
RAW_DIR.mkdir(parents=True, exist_ok=True)

nepal = pd.DataFrame({
    "Year": range(2010, 2025),
    "Country": "Nepal",
    "Country_Code": "NPL",
})

download_status = {}
download_log = {}

print("Download batch:", RUN_ID)

Download batch: 20261007T164500800228Z


In [6]:
for column, code in INDICATORS.items():
    nepal[column] = float("nan")

    url = f"https://api.worldbank.org/v2/country/NPL/indicator/{code}"
    response_file = RAW_DIR / f"{code}.json"

    try:
        response = requests.get(url, params=PARAMS, timeout=60)

        # Received response preserve karo, including API error responses.
        response_file.write_bytes(response.content)
        response.raise_for_status()

        payload = response.json()

        if not (
            isinstance(payload, list)
            and len(payload) == 2
            and isinstance(payload[0], dict)
        ):
            raise ValueError(f"Unexpected API response: {str(payload)[:300]}")

        header, observations = payload

        if int(header.get("pages", 1)) > 1:
            raise ValueError("Response has multiple pages; pagination required.")

        # A valid empty response is different from a failed download.
        if observations is None and int(header.get("total", -1)) == 0:
            observations = []

        if not isinstance(observations, list):
            raise ValueError("API did not return an observations list.")

        years = [int(row["date"]) for row in observations]

        if len(years) != len(set(years)):
            raise ValueError("Duplicate years found in the API response.")

        values = {
            int(row["date"]): row["value"]
            for row in observations
        }

        # Match values by year; unavailable values remain missing.
        nepal[column] = pd.to_numeric(
            nepal["Year"].map(values),
            errors="raise",
        )

        download_status[column] = "OK"
        download_log[column] = {
            "code": code,
            "url": response.url,
            "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
            "source_last_updated": header.get("lastupdated"),
            "raw_file": str(response_file.relative_to(ROOT)),
            "status": "OK",
        }

        n = int(nepal[column].notna().sum())
        print(f"{column}: {n}/15 values available")

    except (requests.RequestException, ValueError, KeyError, TypeError) as error:
        download_status[column] = "DOWNLOAD_FAILED"
        download_log[column] = {
            "code": code,
            "status": "DOWNLOAD_FAILED",
            "error": str(error),
        }

        print(f"{column}: DOWNLOAD FAILED — {error}")

GDP_per_capita: 15/15 values available
GDP_growth: 15/15 values available
Life_expectancy: 15/15 values available
Basic_drinking_water: 15/15 values available
Secondary_school_enrollment: 14/15 values available
Electricity_access: 15/15 values available
Internet_users: 15/15 values available
CO2_per_capita: DOWNLOAD FAILED — Unexpected API response: [{'message': [{'id': '120', 'key': 'Invalid value', 'value': 'The provided parameter value is not valid'}]}]
Forest_area: 14/15 values available


To download co2 per capita from EDGAR/JRC

In [7]:
CO2_CANDIDATE_CODE = "EN.GHG.CO2.PC.CE.AR5"

candidate_url = (
    "https://api.worldbank.org/v2/country/NPL/indicator/"
    f"{CO2_CANDIDATE_CODE}"
)

candidate_response = requests.get(
    candidate_url,
    params=PARAMS,
    timeout=60,
)

candidate_time = datetime.now(timezone.utc)
candidate_stamp = candidate_time.strftime("%Y%m%dT%H%M%S%fZ")

# Original response preserve karo.
candidate_raw = (
    ROOT / "data" / "raw"
    / f"{CO2_CANDIDATE_CODE}_{candidate_stamp}.json"
)
candidate_raw.write_bytes(candidate_response.content)

candidate_response.raise_for_status()
candidate_payload = candidate_response.json()

if not (
    isinstance(candidate_payload, list)
    and len(candidate_payload) == 2
    and isinstance(candidate_payload[0], dict)
    and isinstance(candidate_payload[1], list)
):
    raise ValueError(f"Unexpected API response: {candidate_payload}")

co2_candidate = pd.DataFrame(
    [
        {
            "Year": int(row["date"]),
            "CO2_per_capita_excl_LULUCF": row["value"],
        }
        for row in candidate_payload[1]
    ],
    columns=["Year", "CO2_per_capita_excl_LULUCF"],
)

co2_candidate = (
    co2_candidate
    .set_index("Year")
    .reindex(range(2010, 2025))
    .reset_index()
)

candidate_record = {
    "assignment_code": "EN.ATM.CO2E.PC",
    "candidate_code": CO2_CANDIDATE_CODE,
    "unit": "t CO2e/capita",
    "downloaded_at_utc": candidate_time.isoformat(),
    "url": candidate_response.url,
    "source_last_updated": candidate_payload[0].get("lastupdated"),
    "role": "Candidate alternative series",
}

(ROOT / "data" / "metadata" / f"co2_candidate_{candidate_stamp}.json").write_text(
    json.dumps(candidate_record, indent=2),
    encoding="utf-8",
)

co2_candidate.to_csv(
    ROOT / "data" / "processed" / f"nepal_co2_candidate_{candidate_stamp}.csv",
    index=False,
)

print(
    "Available CO2 observations:",
    co2_candidate["CO2_per_capita_excl_LULUCF"].notna().sum(),
    "/ 15",
)

display(co2_candidate)

Available CO2 observations: 15 / 15


,Year,CO2_per_capita_excl_LULUCF
0,2010,0.175006
1,2011,0.195187
2,2012,0.226080
3,2013,0.228210
4,2014,0.265695
5,2015,0.263161
6,2016,0.385041
7,2017,0.471232
8,2018,0.531555
9,2019,0.473751
